# Phase 2 — Acquisition des données d'activité biologique (cible ERBB2/HER2)

Objectif : récupérer les molécules dont l'activité biologique sur la cible **ERBB2 (HER2)** a été mesurée, pour construire un jeu de données de screening in silico (QSAR).

ERBB2 est choisie car elle relie directement à OncoPrint : c'est la cible du trastuzumab, traitement de référence du sous-type "HER2-enrichi".

**Source de données : PubChem BioAssay** (table "bioactivity concise", via l'API PUG REST). Le plan initial visait ChEMBL, mais son API était en panne (erreurs 500) au moment de l'écriture — PubChem expose les mêmes données d'activité biologique (IC50/AC50, agrégées par gène cible) et reste la référence publique la plus complète après ChEMBL.

In [1]:
import time
import numpy as np
import pandas as pd
import requests

PUG_REST = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
GENE_SYMBOL = "ERBB2"

## 1. Identifier la cible ERBB2 chez l'humain

In [2]:
resp = requests.get(f"{PUG_REST}/gene/genesymbol/{GENE_SYMBOL}/summary/JSON", timeout=30)
resp.raise_for_status()
genes = resp.json()["GeneSummaries"]["GeneSummary"]

human_gene = next(g for g in genes if g["Taxonomy"].startswith("Homo sapiens"))
gene_id = human_gene["GeneID"]
print("Cible retenue :", gene_id, "-", human_gene["Name"])

Cible retenue : 2064 - erb-b2 receptor tyrosine kinase 2


## 2. Récupérer les activités biologiques (table "bioactivity concise")

Cette table agrège, pour un gène cible donné, tous les résultats de tests biologiques publics (AID = identifiant du test, CID = identifiant de la molécule, Activity Outcome = actif/inactif, Activity Value = valeur en µM).

In [3]:
resp = requests.get(f"{PUG_REST}/gene/geneid/{gene_id}/concise/JSON", timeout=60)
resp.raise_for_status()
table = resp.json()["Table"]

columns = table["Columns"]["Column"]
rows = [r["Cell"] for r in table["Row"]]
raw = pd.DataFrame(rows, columns=columns)
print(raw.shape)
raw.head()

(1178, 12)


,AID,SID,CID,Activity Outcome,Target Accession,Activity Name,Activity Qualifier,Activity Value [uM],Assay Name,Assay Type,PubMed ID,RNAi
0,1433,50100089,16007391,Inactive,NP_001005862,,,4.3,Kinase Inhibitor Selectivity Profiling Assay,Other,,
1,1433,50100094,156414,Active,NP_001005862,,,0.087,Kinase Inhibitor Selectivity Profiling Assay,Other,,
2,1433,50100096,9874913,Active,NP_001005862,,,0.043,Kinase Inhibitor Selectivity Profiling Assay,Other,,
3,1433,50100097,3062316,Active,NP_001005862,,,1.4,Kinase Inhibitor Selectivity Profiling Assay,Other,,
4,1433,50100098,6445562,Active,NP_001005862,,,0.5,Kinase Inhibitor Selectivity Profiling Assay,Other,,


## 3. Nettoyage et agrégation par molécule

- On garde les lignes avec un CID et un `Activity Outcome` clair (Active/Inactive).
- Une même molécule peut apparaître dans plusieurs tests (`AID`) : on agrège par CID en gardant la valeur d'activité la plus forte (µM le plus bas = molécule la plus active observée) et le résultat majoritaire.

In [4]:
df = raw[raw["CID"].astype(bool) & raw["Activity Outcome"].isin(["Active", "Inactive"])].copy()
df["CID"] = df["CID"].astype(int)
df["Activity Value [uM]"] = pd.to_numeric(df["Activity Value [uM]"], errors="coerce")

def aggregate_compound(group):
    outcome = group["Activity Outcome"].mode().iloc[0]
    value = group["Activity Value [uM]"].min()  # meilleure activité observée
    return pd.Series({"activity_value_um": value, "outcome": outcome, "n_assays": len(group)})

compounds = df.groupby("CID").apply(aggregate_compound, include_groups=False).reset_index()
compounds["active"] = (compounds["outcome"] == "Active").astype(int)
# pActivity (style pIC50) quand une valeur numérique est disponible
compounds["pactivity"] = -np.log10(compounds["activity_value_um"] * 1e-6)

print(compounds.shape)
print(compounds["active"].value_counts())
compounds.head()

(1070, 6)
active
1    546
0    524
Name: count, dtype: int64


,CID,activity_value_um,outcome,n_assays,active,pactivity
0,1400,NaN,Active,1,1,NaN
1,2051,0.04645,Active,1,1,7.333014
2,3501,NaN,Active,1,1,NaN
3,3536,NaN,Inactive,1,0,NaN
4,3541,NaN,Inactive,1,0,NaN


## 4. Récupérer les SMILES de chaque molécule

Par lots de 150 CID pour rester dans les limites de l'API (max ~5 requêtes/s).

In [5]:
def fetch_smiles(cids, batch_size=150, pause=0.25):
    results = {}
    for i in range(0, len(cids), batch_size):
        batch = cids[i:i + batch_size]
        ids = ",".join(str(c) for c in batch)
        url = f"{PUG_REST}/compound/cid/{ids}/property/SMILES/JSON"
        resp = requests.get(url, timeout=60)
        resp.raise_for_status()
        for prop in resp.json()["PropertyTable"]["Properties"]:
            results[prop["CID"]] = prop["SMILES"]
        time.sleep(pause)
    return results

smiles_map = fetch_smiles(compounds["CID"].tolist())
compounds["canonical_smiles"] = compounds["CID"].map(smiles_map)
compounds = compounds.dropna(subset=["canonical_smiles"])
print(compounds.shape)
compounds.head()

(1070, 7)


,CID,activity_value_um,outcome,n_assays,active,pactivity,canonical_smiles
0,1400,NaN,Active,1,1,NaN,CC1=CC=C(C=C1)C2=NN(C3=NC=NC(=C23)N)C(C)(C)C
1,2051,0.04645,Active,1,1,7.333014,COC1=C(C=C2C(=C1)C(=NC=N2)NC3=CC(=CC=C3)Cl)OC
2,3501,NaN,Active,1,1,NaN,CN1C2=CC=CC=C2C3=C4C(=C5C6=CC=CC=C6N(C5=C31)CC...
3,3536,NaN,Inactive,1,0,NaN,C1=CC=NC(=C1)NS(=O)(=O)C2=CC=C(C=C2)N=CC3=C(NC...
4,3541,NaN,Inactive,1,0,NaN,C1=CC2=C(C=CN=C2)C(=C1)S(=O)(=O)NCCNCC=CC3=CC=...


In [6]:
compounds = compounds.rename(columns={"CID": "cid"})
cols = ["cid", "canonical_smiles", "activity_value_um", "pactivity", "outcome", "active", "n_assays"]
compounds[cols].to_csv("../data/raw/erbb2_activities.csv", index=False)
print("Sauvegardé :", len(compounds), "molécules")

Sauvegardé : 1070 molécules
